# HW1: the Bradley-Terry model, from scratch

**RLHF course, session 2 · Pegah Alizadeh** · Reading: Lambert, Chapter 5, [Training a Bradley-Terry reward model](https://rlhfbook.com/c/05-reward-models#training-a-bradley-terry-reward-model)

In Lab 2 you trained a reward model on real language model outputs. It rests on the Bradley-Terry model of paired comparisons. In this homework we study that model where **the truth is known**, and check whether the estimates and their uncertainty behave as theory promises.

Item $i$ has a latent strength $\theta_i$, and

$$P(i \text{ beats } j) = \sigma(\theta_i - \theta_j), \qquad \sigma(u) = \frac{1}{1 + e^{-u}}.$$

| Level | What it asks |
|---|---|
| **Foundation** | Simulate data and fit the model |
| **Core** | Build confidence intervals and test their coverage |
| **Deep** | Derive the Fisher information, and study correlated comparisons |

This homework is **not graded**. Foundation and Core take about an hour; Deep is for those who want to go further.

**How to use it.** Each exercise has a **hint** and a **solution** you can open, and a **check** cell that tells you
whether your code is right. Each question has an **answer** you can open. Try first, then open them: the aim is to
understand, not to finish.

**No GPU is needed:** run it in Google Colab or Kaggle with the default runtime. Write your answers in text cells in the notebook.

**Sources.** The exercises were written for this course, using standard results:

- **The model:** Bradley and Terry (1952), *Biometrika* 39. The RLHF book, Chapter 5, [Training a Bradley-Terry reward model](https://rlhfbook.com/c/05-reward-models#training-a-bradley-terry-reward-model), explains how reward models use it.
- **Bradley-Terry as a logistic regression, and maximum likelihood:** A. Agresti, *Categorical Data Analysis*, in its chapter on paired comparisons. Also S. Truong, A. Haupt and S. Koyejo, [*Machine Learning from Human Preferences*](https://mlhp.stanford.edu), [Section 2.3](https://mlhp.stanford.edu/src/chap2.html#sec-mle) and [Exercise 1.17.6](https://mlhp.stanford.edu/src/chap1.html#bradley-terry-maximum-likelihood).
- **Identifiability:** *Machine Learning from Human Preferences*, [Section 1.9](https://mlhp.stanford.edu/src/chap1.html#sec-identification-rashomon).
- **Fisher information and the Wald interval:** any course on likelihood theory. *Machine Learning from Human Preferences* also uses it in [Section 2.4.2.2](https://mlhp.stanford.edu/src/chap2.html#sec-fisher-information).
- **Correlated comparisons and cluster-robust intervals:** K.-Y. Liang and S. Zeger (1986), *Biometrika* 73, and A. C. Cameron and D. Miller (2015), "A practitioner's guide to cluster-robust inference", *Journal of Human Resources* 50.

## The big picture

In RLHF, people compare two responses and say which is better. A **reward model** turns these comparisons into a score for
each response, and that score is what the model is later trained to increase. The statistical model behind it is
Bradley-Terry.

In this homework, the items are abstract (think of 6 responses, or 6 chess players), and we **choose the true strengths
ourselves**. Because we know the truth, we can check whether estimates and intervals behave as theory promises. This is the
standard way to test a statistical method before using it on real data.

### Key words

| Word | Meaning |
|---|---|
| **Latent strength** $\theta_i$ | An unobserved number for each item. Higher means more likely to win. |
| **Comparison** | One observation: items $i$ and $j$, and $y = 1$ if $i$ won |
| **Maximum likelihood** | The parameter values that make the observed comparisons most probable |
| **Fisher information** | How much the data tell us about the parameters. Its inverse approximates the variance of the estimate. |
| **Wald interval** | Estimate $\pm 1.96 \times$ standard error |
| **Coverage** | The fraction of repeated experiments in which the interval contains the true value |
| **Monte Carlo** | Estimating a quantity by repeating a random simulation many times |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def sigmoid(u):
    # The logistic function: turns a strength difference into a win probability.
    return 1 / (1 + np.exp(-u))


theta_true = np.array([0.0, 0.5, 1.0, -0.5, 1.5, -1.0])   # item 0 is the reference
K = len(theta_true)

### A first look: who beats whom?

Before simulating anything, compute the **true** probability that each item beats each other one. Row $i$, column $j$
is $P(i \text{ beats } j)$.

In [ ]:
P = sigmoid(theta_true[:, None] - theta_true[None, :])
print("strengths:", theta_true)
print(np.round(P, 2))

Three things to check in this table:

- **The diagonal is 0.5:** an item against itself is a coin flip.
- **$P(i \text{ beats } j) + P(j \text{ beats } i) = 1$:** the table is "antisymmetric" around 0.5.
- **Only differences matter:** item 4 ($\theta = 1.5$) beats item 2 ($\theta = 1.0$) with the same probability as item 1
  ($\theta = 0.5$) beats item 0 ($\theta = 0$), because both differences are 0.5.

---
# Foundation

### Question 1: identifiability

Show that $\theta$ and $\theta + c$ give the same likelihood for any constant $c$. What does this mean about what the data can tell us? This is why we fix $\theta_0 = 0$.


<details><summary>Hint</summary>

Write the probability of one comparison for $\theta + c$, and simplify the difference inside $\sigma$.
</details>


<details><summary>Answer</summary>

$\sigma((\theta_i + c) - (\theta_j + c)) = \sigma(\theta_i - \theta_j)$, so the likelihood depends only on differences. The data determine strengths only up to a constant, so only comparisons between items have meaning. Fixing $\theta_0 = 0$ picks one representative, and the others are then differences from item 0.
</details>

### Exercise 1: simulate comparisons

**One comparison, by hand.** Take items 4 and 5: $\theta_4 - \theta_5 = 1.5 - (-1.0) = 2.5$, so item 4 wins with
probability $p = \sigma(2.5) \approx 0.924$. To simulate the outcome, draw $u$ uniformly in $[0, 1]$: item 4 wins
($y = 1$) if $u < p$. So $u = 0.31$ gives $y = 1$, and $u = 0.97$ gives $y = 0$.

`simulate` does this $n$ times at once, with two items picked at random each time.


<details><summary>Hint</summary>

- `rng.integers(0, K, n)` draws $n$ items. To make $j$ different from $i$, add a random shift between 1 and $K - 1$ and
  take the result modulo $K$.
- `rng.random(n) < p` gives $n$ outcomes at once; `.astype(int)` turns them into 0 and 1.
</details>


<details><summary>Solution</summary>

```python
def simulate(theta, n, rng):
    # n random comparisons under Bradley-Terry with true strengths theta.
    # Returns i, j (the two items of each comparison) and y (1 if i won), three arrays of length n.
    K = len(theta)
    i = rng.integers(0, K, n)                  # first item, uniform
    j = (i + rng.integers(1, K, n)) % K        # second item: a shift of 1 to K-1, so always different from i
    p = sigmoid(theta[i] - theta[j])           # P(i wins), from the model
    y = (rng.random(n) < p).astype(int)        # draw the outcomes: i wins with probability p
    return i, j, y
```
</details>

In [ ]:
def simulate(theta, n, rng):
    # TODO: return (i, j, y), three arrays of length n.
    # i, j: two different items, chosen uniformly at random
    # y: 1 if item i wins, with probability sigmoid(theta[i] - theta[j]), and 0 otherwise
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
i, j, y = simulate(theta_true, 50000, rng)
assert (i != j).all()
strongest = (i == 4) & (j == 5)      # theta difference of 2.5
assert abs(y[strongest].mean() - sigmoid(2.5)) < 0.03
print("Exercise 1: correct")

### Does the simulation match the model?

With 50,000 simulated comparisons, the observed win rate of each pair should be close to the true probability above.

In [ ]:
wins = np.zeros((K, K))
games = np.zeros((K, K))
np.add.at(games, (i, j), 1)
np.add.at(wins, (i, j), y)
observed = np.where(games > 0, wins / np.maximum(games, 1), np.nan)
print("Observed win rate of row item against column item:")
print(np.round(observed, 2))
print("Largest gap from the true probabilities:", np.round(np.nanmax(np.abs(observed - P)), 3))

Each pair is compared about 1,700 times here, so each observed rate has a standard error of about
$\sqrt{0.25 / 1700} \approx 0.012$. Gaps of a few hundredths are just sampling noise.

### Exercise 2: Bradley-Terry is a logistic regression

Writing $\beta = (\theta_1, \ldots, \theta_{K-1})$, each comparison is a row $x$ with $+1$ for item $i$ and $-1$ for item $j$, and $P(y = 1) = \sigma(x^\top \beta)$. There is no intercept.

**A tiny example with 3 items**, where item 0 is the reference and has no column:

| Comparison | Column for item 1 | Column for item 2 | $x^\top \beta$ |
|---|---|---|---|
| 1 against 2 | $+1$ | $-1$ | $\theta_1 - \theta_2$ |
| 2 against 0 | $0$ | $+1$ | $\theta_2 - \theta_0 = \theta_2$ |
| 0 against 1 | $-1$ | $0$ | $\theta_0 - \theta_1 = -\theta_1$ |

Each row gives exactly the difference that Bradley-Terry needs. So fitting Bradley-Terry **is** fitting a logistic
regression on this design matrix, and any logistic regression software can do it.

**Newton's method** improves the estimate using the curvature of the log-likelihood. For logistic regression, the
log-likelihood is concave, so Newton's method converges quickly to the unique maximum, usually in a handful of steps.
This algorithm is also called **iteratively reweighted least squares**.

Complete `design_matrix`, then fit the model with Newton's method. The update is

$$\beta \leftarrow \beta + \left(X^\top W X\right)^{-1} X^\top (y - p), \qquad W = \mathrm{diag}\big(p_m (1 - p_m)\big).$$


<details><summary>Hint</summary>

- **Design matrix:** start from an $n \times K$ matrix of zeros, put $+1$ at `(row, i)` and $-1$ at `(row, j)` with
  `np.arange(n)` as the rows, then drop column 0.
- **Newton:** $X^\top W X$ never needs $W$ as a matrix: `(X * w[:, None]).T @ X` multiplies each row by its weight.
  Use `np.linalg.solve` rather than inverting.
</details>


<details><summary>Solution</summary>

```python
def design_matrix(i, j, K):
    # One row per comparison: +1 for item i, -1 for item j, so that x @ theta = theta[i] - theta[j].
    # Returns an (n, K-1) matrix: item 0 has no column, because theta[0] = 0 is fixed.
    n = len(i)
    full = np.zeros((n, K))
    full[np.arange(n), i] += 1                 # +1 in the column of the first item
    full[np.arange(n), j] -= 1                 # -1 in the column of the second item
    return full[:, 1:]                         # drop item 0's column

def fit(X, y, steps=25):
    # Maximum likelihood for the logistic regression P(y = 1) = sigmoid(X @ beta), by Newton's method.
    # Returns beta, the estimated strengths of items 1 to K-1 (item 0 is fixed at 0).
    d = X.shape[1]
    beta = np.zeros(d)                                                    # start: all items equal
    for _ in range(steps):
        p = sigmoid(X @ beta)                                             # current win probabilities
        gradient = X.T @ (y - p)                                          # score: observed minus predicted
        hessian = (X * (p * (1 - p))[:, None]).T @ X + 1e-6 * np.eye(d)   # X^T W X, plus a tiny ridge for safety
        beta = beta + np.linalg.solve(hessian, gradient)                  # Newton step
    return beta
```
</details>

In [ ]:
def design_matrix(i, j, K):
    # TODO: return an (n, K-1) matrix. Row m has +1 in the column of item i[m] and -1 in the
    # column of item j[m], and item 0 has no column, because theta[0] = 0 is fixed.
    raise NotImplementedError

In [ ]:
def fit(X, y, steps=25):
    # TODO: return the maximum likelihood estimate of beta (theta without item 0), using Newton's method.
    # A small ridge, such as 1e-6 * np.eye(d), keeps the matrix invertible on unlucky samples.
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
i, j, y = simulate(theta_true, 20000, rng)
X = design_matrix(i, j, K)
assert X.shape == (20000, K - 1)
beta_hat = fit(X, y)
print("True:     ", theta_true[1:])
print("Estimated:", np.round(beta_hat, 2))
assert np.abs(beta_hat - theta_true[1:]).max() < 0.1
print("Exercise 2: correct")

### Watch Newton's method converge

Here are the estimates after 1, 2, 3, 4 and 5 Newton steps, on the same data:

In [ ]:
print("Your design matrix for 3 comparisons (1 vs 2, 2 vs 0, 0 vs 1), with K = 3:")
print(design_matrix(np.array([1, 2, 0]), np.array([2, 0, 1]), 3))
print()
for steps in range(1, 6):
    print(f"after {steps} steps:", np.round(fit(X, y, steps=steps), 4))
print("true values:  ", theta_true[1:])

Newton's method usually stops changing after 3 or 4 steps. The estimates are close to the truth but not equal to it:
that remaining gap is sampling error, which the next exercises measure.

---
# Core

### Exercise 3: Wald confidence intervals

Asymptotically, $\hat\beta \approx \mathcal{N}\big(\beta, I(\beta)^{-1}\big)$, with Fisher information $I(\beta) = X^\top W X$. Build 95% intervals for each $\beta_k$.

**Where this comes from.** With many independent comparisons, the maximum likelihood estimate is approximately normal,
centered on the truth. Its covariance is the inverse Fisher information, evaluated at the estimate. So:

1. Compute $I(\hat\beta) = X^\top W X$, with $W$ built from the fitted probabilities.
2. Invert it. The **standard error** of $\hat\beta_k$ is the square root of the $k$-th diagonal entry.
3. The interval is $\hat\beta_k \pm 1.96 \times \text{standard error}$.

The Fisher information is the same matrix that appears in your Newton step: the curvature of the log-likelihood.

**A tiny example, with two items.** Item 1 against item 0, $n = 100$ times, and the fitted win probability is
$\hat p = 0.62$. Each row is $x = (+1)$, so $I = \sum_m \hat p(1 - \hat p) = 100 \times 0.62 \times 0.38 = 23.6$. The
standard error is $1 / \sqrt{23.6} \approx 0.21$, so the interval is $\hat\beta_1 \pm 0.41$, with
$\hat\beta_1 = \log(0.62 / 0.38) \approx 0.49$. Your function does the same with a matrix.


<details><summary>Hint</summary>

The Fisher information is the matrix of your Newton step, at $\hat\beta$. The variances are on the diagonal of its
inverse: `np.diag(np.linalg.inv(fisher))`.
</details>


<details><summary>Solution</summary>

```python
def wald_interval(X, beta, z=1.96):
    # 95% Wald intervals for each estimated strength, from the Fisher information at the estimate.
    # Returns (lower, upper), two arrays of length K-1.
    p = sigmoid(X @ beta)                                                     # fitted probabilities
    fisher = (X * (p * (1 - p))[:, None]).T @ X + 1e-6 * np.eye(X.shape[1])   # I = X^T W X, as in Newton's method
    se = np.sqrt(np.diag(np.linalg.inv(fisher)))                              # standard errors: diagonal of I^-1
    return beta - z * se, beta + z * se
```
</details>

In [ ]:
def wald_interval(X, beta, z=1.96):
    # TODO: return (lower, upper), arrays of length d, from the inverse Fisher information.
    raise NotImplementedError

### Your intervals on one dataset

With 500 comparisons, here are the estimates, the intervals, and whether each interval contains the true value:

In [ ]:
rng = np.random.default_rng(13)
i, j, y = simulate(theta_true, 500, rng)
X = design_matrix(i, j, K)
beta_hat = fit(X, y)
lower, upper = wald_interval(X, beta_hat)
assert (lower < upper).all(), "Each lower bound should be below its upper bound."
# Check your answer on the two-item example above: 62 wins of item 1 out of 100.
X_toy = np.ones((100, 1))
y_toy = np.array([1] * 62 + [0] * 38)
low_toy, up_toy = wald_interval(X_toy, np.array([np.log(0.62 / 0.38)]))
assert abs((up_toy[0] - low_toy[0]) / 2 - 1.96 / np.sqrt(23.56)) < 1e-3, "Check the standard error: square root of the diagonal of the inverse."
print("Exercise 3: correct")
for k in range(K - 1):
    covered = lower[k] <= theta_true[k + 1] <= upper[k]
    print(f"item {k + 1}: estimate {beta_hat[k]:6.2f}, interval [{lower[k]:6.2f}, {upper[k]:6.2f}], "
          f"true {theta_true[k + 1]:5.2f}, covered: {covered}")

On one dataset, most intervals should contain the truth, but perhaps not all. A 95% interval is allowed to miss 5% of
the time. Misses can also come together: every estimate is measured relative to item 0, so if item 0 happens to do
unusually well or badly in the data, all the estimates shift at once. Whether it really misses only 5% of the time can't be seen from one dataset: that's what coverage measures.

### Exercise 4: do they really cover 95%?

A confidence level is a claim about repeated sampling, and in simulation we can test it. Estimate the coverage for several sample sizes, then plot it.

**The recipe:** repeat many times: simulate a new dataset, fit the model, build the intervals, and record whether each
contains the true value. The fraction of hits estimates the coverage. If the method is right, it should be close to 0.95.

**Question 2.** How close to 95% is the coverage at small $n$? How many simulations do you need before differences from 95% are more than simulation noise? (Compute the Monte Carlo standard error.)


<details><summary>Hint</summary>

Use the functions you already wrote, in a loop of `sims` repetitions. `(lower <= truth) & (truth <= upper)` is an array of
True/False, one per item; adding it to a running total counts the hits.
</details>


<details><summary>Solution</summary>

```python
def coverage(theta, n, sims, rng):
    # Monte Carlo estimate of the coverage of the Wald interval, with n comparisons per dataset.
    # Returns one fraction per item (1 to K-1): how often its interval contained the true strength.
    K = len(theta)
    hits = np.zeros(K - 1)
    for _ in range(sims):                                   # repeat the whole experiment
        i, j, y = simulate(theta, n, rng)                   # a new dataset
        X = design_matrix(i, j, K)
        beta = fit(X, y)                                    # estimate
        lower, upper = wald_interval(X, beta)               # intervals
        hits += (lower <= theta[1:]) & (theta[1:] <= upper) # 1 for each interval that contains the truth
    return hits / sims
```
</details>


<details><summary>Answer to Question 2</summary>

With $S$ simulations, the coverage estimate is a proportion, so its Monte Carlo standard error is
$\sqrt{0.95 \times 0.05 / S}$, about 0.012 for $S = 300$. Differences from 0.95 smaller than about 0.02 are therefore
not meaningful here. Coverage may fall somewhat below 95% at small $n$, because the normal approximation is less
accurate there, and it should approach 95% as $n$ grows.
</details>

In [ ]:
def coverage(theta, n, sims, rng):
    # TODO: over `sims` simulated datasets, return the fraction of intervals containing the truth,
    # as an array with one entry per item (excluding item 0).
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(2)
sizes = [100, 300, 1000]
results = [coverage(theta_true, n, 300, rng) for n in sizes]
assert all(c.shape == (K - 1,) and ((0 <= c) & (c <= 1)).all() for c in results), "coverage() should return one fraction per item."
assert results[-1].mean() > 0.85, "At n = 1000 the coverage should be close to 0.95. Check the interval and the loop."
print("Exercise 4: correct")
for n, c in zip(sizes, results):
    print(f"n = {n:5d}: mean coverage {c.mean():.3f}, by item {np.round(c, 2)}")
plt.plot(sizes, [c.mean() for c in results], marker="o")
plt.axhline(0.95, linestyle="--", color="gray")
plt.xscale("log")
plt.xlabel("Number of comparisons")
plt.ylabel("Mean coverage")
plt.show()

**How to read the plot.** The dashed line is the nominal 95%. Points above it mean intervals that are a little too wide,
and points below it mean intervals that are too narrow, which is the more dangerous error. Remember that each point is
itself an estimate from a finite number of simulations, so small differences from 0.95 may be noise: Question 2 asks
how small.

---
# Deep

### Question 3: Fisher information

Derive the score and the Fisher information for one comparison, and show that $I(\beta) = \sum_m p_m(1-p_m)\, x_m x_m^\top$.

Then answer: (a) why is $I(\beta)$ singular if we don't fix $\theta_0$? (b) Which comparisons carry the most information? (c) What does that suggest about how to choose which pairs to label?


<details><summary>Hint</summary>

Start from $\ell = y \log p + (1-y)\log(1-p)$ with $p = \sigma(x^\top\beta)$, and use $\sigma' = \sigma(1-\sigma)$.
The Fisher information is the variance of the score.
</details>


<details><summary>Answer to Question 3</summary>

 The log-likelihood of one comparison is $\ell = y \log p + (1-y)\log(1-p)$, with $p = \sigma(x^\top\beta)$. Since $\sigma' = \sigma(1-\sigma)$, the score is $\nabla\ell = (y - p)x$. Then $I = \mathrm{Var}(\nabla\ell) = p(1-p)\,xx^\top$, because $\mathrm{Var}(y) = p(1-p)$. Summing over independent comparisons gives the result.

(a) With all $K$ columns, each row sums to zero, so $x^\top \mathbf{1} = 0$ and $\mathbf{1}$ is in the null space. This is the identifiability problem of Question 1.
(b) $p(1-p)$ is largest at $p = 1/2$, so comparisons between items of similar strength carry the most information.
(c) A good design would favor close pairs, while keeping the comparison graph connected, as in active learning and dueling bandits.
</details>

### Exercise 5: correlated comparisons

Our intervals assume independent comparisons. In real preference data, one annotator gives many labels and annotators differ. Chapter 5 raises a related concern about correlated comparisons.

Simulate this. Each annotator has strengths $\theta + u$, with $u_k \sim \mathcal{N}(0, \tau^2)$ independently per item. Estimate the coverage of the same Wald interval, and of an interval you propose that accounts for annotators.

**Target:** with annotator effects, $\hat\beta$ estimates a population-averaged effect, which is not exactly $\theta_k - \theta_0$. The cell below gives you a reasonable target: `truth(theta)`, the population-averaged log-odds against item 0, attenuated as in a marginal logistic model.

**Why correlation matters.** Labels from the same annotator share that annotator's tastes, so they aren't independent
pieces of information. Ten labels from one person tell us about that person, not ten times as much about the population.

<details><summary>Hint: the design effect</summary>

For an average of $n$ observations in groups of $m$, with correlation $\rho$ between observations of the same group,
the variance is inflated by about $1 + (m - 1)\rho$, compared with independent data. This factor is called the
**design effect**, and $n / (1 + (m - 1)\rho)$ is the **effective sample size**.
</details>

<details><summary>Hint: an interval that accounts for annotators</summary>

A **cluster-robust (sandwich) variance** keeps the inverse Fisher information as the "bread", but replaces the "meat" with
the sum over annotators of $s_a s_a^\top$, where $s_a$ is the total score (gradient of the log-likelihood) of annotator
$a$'s labels. An alternative is a **bootstrap that resamples whole annotators**.
</details>

**Question 4.** How does coverage change with $\tau$, and with the number of labels per annotator for a fixed total? What is the effective sample size when an annotator gives many labels? Why does this matter for real preference data, where one annotator often labels many pairs?


<details><summary>Solution</summary>

```python
def simulate_annotators(theta, n_annotators, per_annotator, tau, rng):
    # Comparisons from several annotators, each with their own tastes: strengths theta + u, u ~ Normal(0, tau^2).
    # Returns i, j, y as in simulate(), and the annotator of each comparison.
    K = len(theta)
    parts = []
    for a in range(n_annotators):
        strengths = theta + rng.normal(0, tau, K)   # this annotator's strengths, shared by all their labels
        i, j, y = simulate(strengths, per_annotator, rng)
        parts.append((i, j, y, np.full(per_annotator, a)))
    return tuple(np.concatenate(values) for values in zip(*parts))


def coverage_correlated(theta, n_annotators, per_annotator, tau, sims, rng):
    # Coverage of the usual Wald interval when the comparisons come from annotators (so they are correlated).
    K = len(theta)
    hits = np.zeros(K - 1)
    for _ in range(sims):
        i, j, y, _ = simulate_annotators(theta, n_annotators, per_annotator, tau, rng)
        X = design_matrix(i, j, K)
        beta = fit(X, y)
        lower, upper = wald_interval(X, beta)
        hits += (lower <= truth(theta)) & (truth(theta) <= upper)
    return hits / sims


def cluster_interval(X, y, beta, annotator, z=1.96):
    """Cluster-robust sandwich: bread = inverse Fisher, meat = sum of squared annotator-level scores."""
    p = sigmoid(X @ beta)
    bread = np.linalg.inv((X * (p * (1 - p))[:, None]).T @ X + 1e-6 * np.eye(X.shape[1]))   # the usual I^-1
    meat = np.zeros_like(bread)
    for a in np.unique(annotator):
        keep = annotator == a                                  # all labels of annotator a
        score = X[keep].T @ (y[keep] - p[keep])                # their total score: one vector per annotator
        meat += np.outer(score, score)                         # treat each annotator, not each label, as independent
    se = np.sqrt(np.diag(bread @ meat @ bread))                # sandwich variance
    return beta - z * se, beta + z * se


def coverage_cluster(theta, n_annotators, per_annotator, tau, sims, rng):
    # Coverage of the cluster-robust interval, on the same kind of data as coverage_correlated().
    K = len(theta)
    hits = np.zeros(K - 1)
    for _ in range(sims):
        i, j, y, annotator = simulate_annotators(theta, n_annotators, per_annotator, tau, rng)
        X = design_matrix(i, j, K)
        beta = fit(X, y)
        lower, upper = cluster_interval(X, y, beta, annotator)
        hits += (lower <= truth(theta)) & (truth(theta) <= upper)
    return hits / sims
```
</details>


<details><summary>Answer to Question 4</summary>

 With $\tau > 0$, labels from one annotator share the same $u$, so they are positively correlated, and the Wald interval is too narrow: it counts the labels as if each were independent. Coverage falls as $\tau$ grows and as more labels come from fewer annotators. Adding labels per annotator then helps less and less, because the number of annotators limits precision. A cluster-robust sandwich or an annotator-level bootstrap restores much of the coverage, but it needs enough annotators to work well. In real preference data, where each annotator labels many pairs, the number of labels overstates how much information we have about the population.
</details>

In [ ]:
def truth(theta, tau=None, draws=200000, seed=123):
    """The population-averaged log-odds of each item against item 0, found by Monte Carlo."""
    tau = TAU if tau is None else tau
    noise = np.random.default_rng(seed).normal(0, tau, (draws, 1)) * np.sqrt(2)
    p = sigmoid(theta[1:] - theta[0] + noise).mean(axis=0)
    return np.log(p / (1 - p))


TAU = 1.0
print("Target:", np.round(truth(theta_true), 3))

In [ ]:
def simulate_annotators(theta, n_annotators, per_annotator, tau, rng):
    # TODO: each annotator has their own strengths: theta + u, with u ~ Normal(0, tau^2) for each item.
    # They make per_annotator comparisons. Return (i, j, y, annotator), each of the total length.
    raise NotImplementedError


def coverage_correlated(theta, n_annotators, per_annotator, tau, sims, rng):
    # TODO: as coverage(), using simulate_annotators and the same Wald interval.
    raise NotImplementedError


def cluster_interval(X, y, beta, annotator, z=1.96):
    # TODO: propose an interval that accounts for the annotators, for example a cluster-robust
    # sandwich estimator or a bootstrap that resamples whole annotators.
    raise NotImplementedError


def coverage_cluster(theta, n_annotators, per_annotator, tau, sims, rng):
    # TODO: coverage of your proposed interval.
    raise NotImplementedError

In [ ]:
# Check the shapes on a small run first.
i_c, j_c, y_c, a_c = simulate_annotators(theta_true, 4, 5, TAU, np.random.default_rng(0))
assert len(i_c) == len(a_c) == 20 and set(a_c) == {0, 1, 2, 3}, "simulate_annotators: 4 annotators x 5 labels = 20 rows."
print("Exercise 5: shapes correct")

rng = np.random.default_rng(3)
for n_annotators, per_annotator in [(100, 10), (20, 50), (10, 100)]:
    naive = coverage_correlated(theta_true, n_annotators, per_annotator, TAU, 200, rng)
    robust = coverage_cluster(theta_true, n_annotators, per_annotator, TAU, 200, rng)
    print(f"{n_annotators:3d} annotators x {per_annotator:3d} labels: Wald {naive.mean():.3f}, cluster-robust {robust.mean():.3f}")

## What you learned

- **Bradley-Terry** gives each item a strength, and only **differences** between strengths are identified
- **Fitting it is a logistic regression** on a design matrix with $+1$ and $-1$, solved by Newton's method
- **The Fisher information** gives standard errors, and **Wald intervals** that cover about 95% when comparisons are independent
- **Coverage can be tested by simulation,** as long as you account for Monte Carlo error
- **Correlated comparisons,** such as many labels from one annotator, make naive intervals too narrow. This matters for
  real preference data, where one annotator often labels many pairs.

**Going further (optional):** use `truth` with care. Is the target exactly right when comparisons don't involve item 0? Try a mixed-effects model, or study how many annotators the sandwich estimator needs.